In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import pypsa
import xlsxwriter
import tz_pypsa
import tz_pypsa.wrangle as wrangle
import pandas as pd
# import tz_solve
import plotly.express as px
import plotly.graph_objects as go
from tz_pypsa.model import Model
from tz_pypsa.utils import get_examples
import os
import glob

In [53]:
n = pypsa.Network()
n.import_from_netcdf("India_hourly_matching_CFE80_2030.nc")

INFO:pypsa.io:Imported network India_hourly_matching_CFE80_2030.nc has buses, carriers, generators, links, loads, storage_units


In [54]:
hourly_df = wrangle.transform_visualiser_hourly_output(n)

In [11]:
hourly_df[(hourly_df['Type'] =='Generation') & (hourly_df['Node'] == 'JPN04') & (hourly_df['Tech'] == 'Batteries')].head(50)

,snapshot,Node,Tech,Value,Type,Node_Destination,HourOfDay,DayOfMonth,Month,Year,Hour8760,long_name,BusType


In [2]:
# ─── USER CONFIG ───────────────────────────────────────────────────────────────
# Path where all your .nc files live (e.g. configs["paths"]["..."])
INPUT_DIR = "G:/Shared drives/Analysis/01 Projects/2024/Google 24 7 CFE/04. Japan/02. Data & Results/Outputs/SingleNodeVsMultiNode Testing"

def list_nc_files(directory):
    """
    Return a sorted list of all .nc file‐paths under `directory`.
    """
    pattern = os.path.join(directory, "*.nc")
    files = sorted(glob.glob(pattern))
    return files

In [3]:
def postprocess_nc_list(nc_files):
    """
    Loop over every single file‐path in `nc_files`, run your PyPSA import + wrangle steps,
    collect the resulting hourly‐DataFrame for each file, then concatenate them all.

    Returns:
        combined_df (pd.DataFrame): one big DataFrame containing all per‐file results.
    """
    all_hourly_dfs = []

    for nc_path in nc_files:
        fname = os.path.basename(nc_path)
        print(f"▶ Processing {fname} …")

        n = pypsa.Network()
        n.import_from_netcdf(nc_path)

        hourly_df = wrangle.transform_visualiser_hourly_output(n)

        # 3) (Optional) Tag every row with the source filename, so you know
        #    which piece of data came from which .nc file later on:
        hourly_df["Scenario"] = fname
        hourly_df = hourly_df[hourly_df['BusType'] == 'Greenfield']  # Filter for Greenfield buses

        # 4) Append to our list
        all_hourly_dfs.append(hourly_df)

        # 5) (Good practice) close the network’s internal file handle if needed
        #    (PyPSA’s Network() doesn’t strictly require a .close(), but if your
        #     pipeline opens files, you can always delete or garbage‐collect `n`.)
        del n

    # After looping through all files, concatenate into one big DataFrame:
    if not all_hourly_dfs:
        # If no files or something failed, return an empty DataFrame
        return pd.DataFrame()

    combined_df = pd.concat(all_hourly_dfs, axis=0, ignore_index=True)
    return combined_df

In [4]:
nc_files = list_nc_files(INPUT_DIR)

# 2) Run the postprocessing loop—this returns one large DataFrame
concat_df = postprocess_nc_list(nc_files)

# 3) Quick sanity check: Show how many rows & columns we ended up with
print(f"✔️  Combined DataFrame shape: {concat_df.shape}")

▶ Processing 1_hourly_matching_CFE60_2030.nc …


INFO:pypsa.io:Imported network 1_hourly_matching_CFE60_2030.nc has buses, carriers, generators, links, loads, storage_units


▶ Processing 1_hourly_matching_CFE70_2030.nc …


INFO:pypsa.io:Imported network 1_hourly_matching_CFE70_2030.nc has buses, carriers, generators, links, loads, storage_units


▶ Processing 1_hourly_matching_CFE80_2030.nc …


...

▶ Processing 9_hourly_matching_CFE70_2030.nc …


INFO:pypsa.io:Imported network 9_hourly_matching_CFE70_2030.nc has buses, carriers, generators, links, loads, storage_units


▶ Processing 9_hourly_matching_CFE80_2030.nc …


INFO:pypsa.io:Imported network 9_hourly_matching_CFE80_2030.nc has buses, carriers, generators, links, loads, storage_units


✔️  Combined DataFrame shape: (3074760, 16)


In [49]:
hourly_df = concat_df[(concat_df['Type'] =='Storage') | (concat_df['Type'] == 'Generation') ]

In [51]:
hourly_df.to_csv('testing.csv', index=False)